In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [3]:
#!/usr/bin/env python3
"""
Spanish Spelling Correction System - Kaggle Version
A complete system for evaluating Spanish spelling correction using fastText embeddings.
"""

import pandas as pd
import numpy as np
from typing import List, Tuple, Dict, Set
import re
from collections import defaultdict
import os
import sys
import pickle
import gzip
import requests
import zipfile
from io import BytesIO
import tempfile
import warnings
warnings.filterwarnings('ignore')

# Install and import fasttext
import subprocess
import importlib

def install_package(package):
    subprocess.check_call([sys.executable, "-m", "pip", "install", package, "--quiet"])

try:
    import fasttext
    import fasttext.util
except ImportError:
    print("Installing fasttext...")
    install_package("fasttext")
    import fasttext
    import fasttext.util

from dataclasses import dataclass

@dataclass
class CorrectionCandidate:
    """Data class for spelling correction candidates"""
    word: str
    edit_distance: int
    embedding_similarity: float
    frequency_score: float
    total_score: float

class SpanishSpellingCorrector:
    """Complete Spanish spelling correction system using fastText embeddings"""
    
    def __init__(self, 
                 vocab_file: str = None,
                 fasttext_model_path: str = None,
                 max_edit_distance: int = 2,
                 alpha: float = 0.5,  # Weight for embedding similarity
                 beta: float = 0.3,   # Weight for frequency
                 gamma: float = 0.2): # Weight for edit distance penalty
        
        self.max_edit_distance = max_edit_distance
        self.alpha = alpha
        self.beta = beta
        self.gamma = gamma
        self.vocab = set()
        self.word_frequencies = {}
        self.ft_model = None
        self.fasttext_model_path = fasttext_model_path
        
        # Spanish alphabet including diacritics
        self.spanish_chars = set('abcdefghijklmnopqrstuvwxyzáéíóúüñ')
        
        print("Initializing Spanish Spelling Corrector...")
        self._load_fasttext_model()
        if vocab_file and os.path.exists(vocab_file):
            self._load_vocabulary(vocab_file)
        else:
            self._create_default_vocabulary()
    
    def _download_fasttext_model(self):
        """Download fastText Spanish model if not available"""
        print("Downloading fastText Spanish model (this may take a while)...")
        try:
            # Download compressed Spanish model
            fasttext.util.download_model('es', if_exists='ignore')
            return "cc.es.300.bin"
        except Exception as e:
            print(f"Error downloading fastText model: {e}")
            # Try alternative download method
            try:
                model_url = "https://dl.fbaipublicfiles.com/fasttext/vectors-crawl/cc.es.300.bin.gz"
                print(f"Downloading from {model_url}")
                response = requests.get(model_url, stream=True)
                with open("cc.es.300.bin.gz", "wb") as f:
                    for chunk in response.iter_content(chunk_size=8192):
                        f.write(chunk)
                
                # Extract the model
                with gzip.open("cc.es.300.bin.gz", "rb") as f_in:
                    with open("cc.es.300.bin", "wb") as f_out:
                        f_out.write(f_in.read())
                
                return "cc.es.300.bin"
            except Exception as e2:
                print(f"Alternative download failed: {e2}")
                print("Using reduced model approach...")
                return self._create_minimal_model()
    
    def _create_minimal_model(self):
        """Create a minimal model for demonstration purposes"""
        print("Creating minimal fastText model...")
        # Create a simple word2vec-like model using basic Spanish words
        try:
            # Try to use a smaller pre-trained model or create dummy vectors
            model = fasttext.train_unsupervised(
                input="dummy_text.txt" if os.path.exists("dummy_text.txt") else "/dev/null",
                model='skipgram',
                dim=100,
                epoch=1,
                minCount=1
            )
            model.save_model("minimal_es_model.bin")
            return "minimal_es_model.bin"
        except:
            return None
    
    def _load_fasttext_model(self):
        """Load fastText Spanish embeddings model"""
        if self.fasttext_model_path and os.path.exists(self.fasttext_model_path):
            model_path = self.fasttext_model_path
        elif os.path.exists("cc.es.300.bin"):
            model_path = "cc.es.300.bin"
        else:
            model_path = self._download_fasttext_model()
        
        if not model_path:
            print("Warning: Could not load fastText model. Using simplified approach.")
            self.ft_model = None
            return
        
        try:
            print(f"Loading fastText model from {model_path}")
            self.ft_model = fasttext.load_model(model_path)
            print("fastText model loaded successfully")
        except Exception as e:
            print(f"Error loading fastText model: {e}")
            print("Continuing without fastText embeddings...")
            self.ft_model = None
    
    def _create_default_vocabulary(self):
        """Create a default Spanish vocabulary"""
        print("Creating default Spanish vocabulary...")
        
        if self.ft_model:
            # Get vocabulary from fastText model
            vocab_words = self.ft_model.get_words()
            spanish_vocab = set()
            for word in vocab_words[:50000]:  # Take top 50k words
                if (2 <= len(word) <= 20 and 
                    all(c.lower() in self.spanish_chars or c in '-' for c in word) and
                    any(c.isalpha() for c in word)):
                    spanish_vocab.add(word.lower())
        else:
            # Use a basic Spanish vocabulary
            spanish_vocab = {
                'el', 'la', 'de', 'que', 'y', 'es', 'en', 'un', 'se', 'no', 'te', 'lo', 'le',
                'da', 'su', 'por', 'son', 'con', 'para', 'al', 'una', 'ser', 'del', 'los',
                'casa', 'agua', 'tiempo', 'año', 'día', 'hombre', 'mujer', 'niño', 'niña',
                'trabajo', 'vida', 'mundo', 'país', 'problema', 'mano', 'lugar', 'caso',
                'parte', 'grupo', 'empresa', 'gobierno', 'persona', 'estado', 'momento',
                'hacer', 'tener', 'estar', 'poder', 'decir', 'todo', 'ir', 'saber', 'ver',
                'dar', 'llegar', 'pasar', 'deber', 'poner', 'parecer', 'quedar', 'creer',
                'hablar', 'llevar', 'dejar', 'seguir', 'encontrar', 'llamar', 'venir',
                'pensar', 'salir', 'volver', 'tomar', 'conocer', 'vivir', 'sentir', 'tratar',
                'mirar', 'contar', 'empezar', 'esperar', 'buscar', 'existir', 'entrar',
                'trabajar', 'escribir', 'perder', 'producir', 'ocurrir', 'entender',
                'pedir', 'recibir', 'recordar', 'terminar', 'permitir', 'aparecer',
                'conseguir', 'comenzar', 'servir', 'sacar', 'necesitar', 'mantener',
                'resultar', 'leer', 'caer', 'cambiar', 'presentar', 'crear', 'abrir',
                'considerar', 'oír', 'acabar', 'convertir', 'ganar', 'formar', 'traer',
                'partir', 'morir', 'aceptar', 'realizar', 'suponer', 'comprender', 'lograr'
            }
        
        self.vocab = spanish_vocab
        print(f"Created vocabulary with {len(self.vocab)} words")
        
        # Create simple frequency scores
        for i, word in enumerate(list(self.vocab)):
            self.word_frequencies[word] = 1.0 / (i + 1)
    
    def _load_vocabulary(self, vocab_file: str):
        """Load vocabulary from file"""
        print(f"Loading vocabulary from {vocab_file}")
        try:
            with open(vocab_file, 'r', encoding='utf-8') as f:
                for line_num, line in enumerate(f, 1):
                    line = line.strip()
                    if line:
                        parts = line.split('\t')
                        word = parts[0].lower()
                        freq = float(parts[1]) if len(parts) > 1 else 1e-6
                        self.vocab.add(word)
                        self.word_frequencies[word] = freq
            print(f"Loaded {len(self.vocab)} words from vocabulary")
        except Exception as e:
            print(f"Error loading vocabulary: {e}")
            self._create_default_vocabulary()
    
    def edit_distance(self, s1: str, s2: str) -> int:
        """Compute Levenshtein edit distance between two strings"""
        if len(s1) < len(s2):
            return self.edit_distance(s2, s1)
        
        if len(s2) == 0:
            return len(s1)
        
        previous_row = list(range(len(s2) + 1))
        for i, c1 in enumerate(s1):
            current_row = [i + 1]
            for j, c2 in enumerate(s2):
                insertions = previous_row[j + 1] + 1
                deletions = current_row[j] + 1
                substitutions = previous_row[j] + (c1 != c2)
                current_row.append(min(insertions, deletions, substitutions))
            previous_row = current_row
        
        return previous_row[-1]
    
    def generate_candidates(self, word: str) -> List[str]:
        """Generate spelling correction candidates within edit distance threshold"""
        word = word.lower()
        candidates = set()
        
        # Add exact matches if in vocabulary
        if word in self.vocab:
            candidates.add(word)
        
        # Generate candidates by edit operations
        candidates.update(self._generate_edits_1(word))
        if self.max_edit_distance > 1:
            candidates.update(self._generate_edits_2(word))
        
        # Filter candidates that are in vocabulary
        valid_candidates = []
        for candidate in candidates:
            if candidate in self.vocab:
                edit_dist = self.edit_distance(word, candidate)
                if edit_dist <= self.max_edit_distance:
                    valid_candidates.append(candidate)
        
        # If no valid candidates found, try approximate matching with vocabulary
        if not valid_candidates:
            valid_candidates = self._find_approximate_matches(word)
        
        return valid_candidates
    
    def _generate_edits_1(self, word: str) -> Set[str]:
        """Generate all possible edits with distance 1"""
        letters = self.spanish_chars
        splits = [(word[:i], word[i:]) for i in range(len(word) + 1)]
        
        deletes = [L + R[1:] for L, R in splits if R]
        transposes = [L + R[1] + R[0] + R[2:] for L, R in splits if len(R) > 1]
        replaces = [L + c + R[1:] for L, R in splits if R for c in letters]
        inserts = [L + c + R for L, R in splits for c in letters]
        
        return set(deletes + transposes + replaces + inserts)
    
    def _generate_edits_2(self, word: str) -> Set[str]:
        """Generate all possible edits with distance 2"""
        edits1 = self._generate_edits_1(word)
        edits2 = set()
        for e1 in list(edits1)[:1000]:  # Limit to prevent memory issues
            edits2.update(self._generate_edits_1(e1))
        return edits2
    
    def _find_approximate_matches(self, word: str, max_candidates: int = 10) -> List[str]:
        """Find approximate matches when no exact edit candidates exist"""
        candidates = []
        for vocab_word in list(self.vocab)[:5000]:  # Limit search for performance
            if abs(len(vocab_word) - len(word)) <= 2:
                edit_dist = self.edit_distance(word, vocab_word)
                if edit_dist <= self.max_edit_distance:
                    candidates.append((vocab_word, edit_dist))
        
        # Sort by edit distance and frequency
        candidates.sort(key=lambda x: (x[1], -self.word_frequencies.get(x[0], 0)))
        return [c[0] for c in candidates[:max_candidates]]
    
    def cosine_similarity(self, vec1: np.ndarray, vec2: np.ndarray) -> float:
        """Compute cosine similarity between two vectors"""
        norm1 = np.linalg.norm(vec1)
        norm2 = np.linalg.norm(vec2)
        if norm1 == 0 or norm2 == 0:
            return 0.0
        return np.dot(vec1, vec2) / (norm1 * norm2)
    
    def correct_word(self, word: str) -> str:
        """Correct a single misspelled word"""
        word_lower = word.lower()
        
        # If word is already correct, return it
        if word_lower in self.vocab:
            return word_lower
        
        # Generate candidates
        candidates = self.generate_candidates(word_lower)
        if not candidates:
            return word_lower  # Return original if no candidates
        
        # Score candidates
        scored_candidates = []
        for candidate in candidates:
            
            # Compute similarity scores
            if self.ft_model:
                input_embedding = self.ft_model.get_word_vector(word_lower)
                candidate_embedding = self.ft_model.get_word_vector(candidate)
                embedding_sim = self.cosine_similarity(input_embedding, candidate_embedding)
            else:
                # Simple character-based similarity if no embeddings
                embedding_sim = 1.0 - (self.edit_distance(word_lower, candidate) / max(len(word_lower), len(candidate)))
            
            frequency_score = self.word_frequencies.get(candidate, 1e-8)
            edit_dist = self.edit_distance(word_lower, candidate)
            edit_penalty = 1.0 / (1.0 + edit_dist)
            
            # Normalize frequency score (log scale)
            freq_score = np.log(frequency_score + 1e-8) + 20  # Shift to positive
            freq_score = max(0, freq_score) / 20.0  # Normalize to [0,1]
            
            # Combined score
            total_score = (self.alpha * embedding_sim + 
                          self.beta * freq_score + 
                          self.gamma * edit_penalty)
            
            scored_candidates.append(CorrectionCandidate(
                word=candidate,
                edit_distance=edit_dist,
                embedding_similarity=embedding_sim,
                frequency_score=freq_score,
                total_score=total_score
            ))
        
        # Return best candidate
        if scored_candidates:
            best_candidate = max(scored_candidates, key=lambda x: x.total_score)
            return best_candidate.word
        
        return word_lower
    
    def evaluate_dataset(self, dataset_path: str) -> Dict:
        """Evaluate the spelling correction system on a dataset"""
        print(f"Loading dataset from {dataset_path}")
        
        try:
            # Try reading as TSV first
            df = pd.read_csv(dataset_path, sep='\t', encoding='utf-8').head(1000)
        except:
            try:
                # Try reading as CSV
                df = pd.read_csv(dataset_path, encoding='utf-8').head(1000)
            except Exception as e:
                print(f"Error loading dataset: {e}")
                return {}
        
        print(f"Dataset loaded: {len(df)} rows")
        print(f"Columns: {list(df.columns)}")
        
        # Ensure required columns exist
        required_cols = ['input', 'target']
        for col in required_cols:
            if col not in df.columns:
                print(f"ERROR: Required column '{col}' not found in dataset")
                return {}
        
        results = []
        correct_predictions = 0
        
        print("Processing corrections...")
        for idx, row in df.iterrows():
            input_word = str(row['input']).strip()
            target_word = str(row['target']).strip()
            
            # Get prediction
            predicted = self.correct_word(input_word)
            is_correct = predicted.lower() == target_word.lower()
            
            if is_correct:
                correct_predictions += 1
            
            result = {
                'input': input_word,
                'predicted': predicted,
                'target': target_word,
                'correct': is_correct,
                'source_rule': row.get('source_rule', 'unknown'),
                'edit_count': row.get('edit_count', -1),
                'freq_bucket': row.get('freq_bucket', -1)
            }
            results.append(result)
            
            if (idx + 1) % 100 == 0:
                print(f"Processed {idx + 1}/{len(df)} rows - Accuracy so far: {correct_predictions/(idx+1):.3f}")
        
        # Calculate overall accuracy
        overall_accuracy = correct_predictions / len(results)
        
        # Calculate accuracy by source_rule
        rule_accuracy = defaultdict(lambda: {'correct': 0, 'total': 0})
        for result in results:
            rule = result['source_rule']
            rule_accuracy[rule]['total'] += 1
            if result['correct']:
                rule_accuracy[rule]['correct'] += 1
        
        # Calculate accuracy by freq_bucket
        bucket_accuracy = defaultdict(lambda: {'correct': 0, 'total': 0})
        for result in results:
            bucket = result['freq_bucket']
            bucket_accuracy[bucket]['total'] += 1
            if result['correct']:
                bucket_accuracy[bucket]['correct'] += 1
        
        evaluation_results = {
            'overall_accuracy': overall_accuracy,
            'total_samples': len(results),
            'correct_predictions': correct_predictions,
            'detailed_results': results,
            'rule_breakdown': {rule: acc['correct']/acc['total'] if acc['total'] > 0 else 0
                             for rule, acc in rule_accuracy.items()},
            'bucket_breakdown': {bucket: acc['correct']/acc['total'] if acc['total'] > 0 else 0
                               for bucket, acc in bucket_accuracy.items()}
        }
        
        return evaluation_results
    
    def save_results(self, results: Dict, output_path: str):
        """Save evaluation results to files"""
        # Save detailed results as CSV
        detailed_df = pd.DataFrame(results['detailed_results'])
        csv_path = output_path.replace('.txt', '_detailed.csv')
        detailed_df.to_csv(csv_path, index=False, encoding='utf-8')
        
        # Save summary results
        with open(output_path, 'w', encoding='utf-8') as f:
            f.write("Spanish Spelling Correction System - Evaluation Results\n")
            f.write("=" * 60 + "\n\n")
            
            f.write(f"Overall Accuracy: {results['overall_accuracy']:.4f}\n")
            f.write(f"Correct Predictions: {results['correct_predictions']}/{results['total_samples']}\n\n")
            
            f.write("Accuracy by Source Rule:\n")
            f.write("-" * 30 + "\n")
            for rule, acc in results['rule_breakdown'].items():
                f.write(f"{rule:15}: {acc:.4f}\n")
            
            f.write("\nAccuracy by Frequency Bucket:\n")
            f.write("-" * 30 + "\n")
            for bucket, acc in results['bucket_breakdown'].items():
                f.write(f"Bucket {bucket:2}: {acc:.4f}\n")
        
        print(f"Results saved to {output_path}")
        print(f"Detailed results saved to {csv_path}")

# Example usage for Kaggle
def run_evaluation(dataset_path, vocab_path=None, output_path="correction_results.txt"):
    """Run the spelling correction evaluation"""
    
    # Initialize corrector
    corrector = SpanishSpellingCorrector(
        vocab_file=vocab_path,
        max_edit_distance=2,
        alpha=0.5,
        beta=0.3,
        gamma=0.2
    )
    
    # Evaluate on dataset
    results = corrector.evaluate_dataset(dataset_path)
    
    if results:
        # Print results
        print("\n" + "="*60)
        print("EVALUATION RESULTS")
        print("="*60)
        print(f"Overall Accuracy: {results['overall_accuracy']:.4f}")
        print(f"Correct: {results['correct_predictions']}/{results['total_samples']}")
        
        print("\nAccuracy by Source Rule:")
        for rule, acc in results['rule_breakdown'].items():
            print(f"  {rule:15}: {acc:.4f}")
        
        print("\nAccuracy by Frequency Bucket:")
        for bucket, acc in results['bucket_breakdown'].items():
            print(f"  Bucket {bucket:2}: {acc:.4f}")
        
        # Save results
        corrector.save_results(results, output_path)
        return results
    else:
        print("ERROR: Could not evaluate dataset")
        return None

# For Kaggle execution - uncomment and modify paths as needed
if __name__ == "__main__":
    # Example usage
    dataset_file = "/kaggle/input/data-synthetic-noise/train.tsv"  # Update with actual path
    vocab_file = None  # "/kaggle/input/your-vocab/vocab.txt"  # Update if you have vocab file
    
    results = run_evaluation(dataset_file, vocab_file, "/kaggle/working/spelling_correction_results.txt")

Initializing Spanish Spelling Corrector...
Loading fastText model from cc.es.300.bin
fastText model loaded successfully
Creating default Spanish vocabulary...
ERROR! Session/line number was not unique in database. History logging moved to new session 29
Created vocabulary with 38032 words
Loading dataset from /kaggle/input/data-synthetic-noise/train.tsv
Dataset loaded: 1000 rows
Columns: ['input', 'target', 'source_rule', 'freq', 'edit_count', 'seed', 'freq_bucket']
Processing corrections...
Processed 100/1000 rows - Accuracy so far: 0.330
Processed 200/1000 rows - Accuracy so far: 0.315
Processed 300/1000 rows - Accuracy so far: 0.330
Processed 400/1000 rows - Accuracy so far: 0.323
Processed 500/1000 rows - Accuracy so far: 0.332
Processed 600/1000 rows - Accuracy so far: 0.327
Processed 700/1000 rows - Accuracy so far: 0.331
Processed 800/1000 rows - Accuracy so far: 0.331
Processed 900/1000 rows - Accuracy so far: 0.327
Processed 1000/1000 rows - Accuracy so far: 0.319

EVALUATION 

In [2]:
# #!/usr/bin/env python3
# """
# Spanish Spelling Correction System
# A complete, deployable system for evaluating Spanish spelling correction using fastText embeddings.
# """

# import pandas as pd
# import numpy as np
# from typing import List, Tuple, Dict, Set
# import re
# from collections import defaultdict
# import os
# import sys
# import argparse
# from dataclasses import dataclass
# import pickle
# import gzip
# import requests
# import zipfile
# from io import BytesIO
# import tempfile
# import warnings
# warnings.filterwarnings('ignore')

# try:
#     import fasttext
#     import fasttext.util
# except ImportError:
#     print("ERROR: fasttext not installed. Install with: pip install fasttext")
#     sys.exit(1)

# @dataclass
# class CorrectionCandidate:
#     """Data class for spelling correction candidates"""
#     word: str
#     edit_distance: int
#     embedding_similarity: float
#     frequency_score: float
#     total_score: float

# class SpanishSpellingCorrector:
#     """Complete Spanish spelling correction system using fastText embeddings"""
    
#     def __init__(self, 
#                  vocab_file: str = None,
#                  fasttext_model_path: str = "cc.es.300.bin",
#                  max_edit_distance: int = 2,
#                  alpha: float = 0.5,  # Weight for embedding similarity
#                  beta: float = 0.3,   # Weight for frequency
#                  gamma: float = 0.2): # Weight for edit distance penalty
        
#         self.max_edit_distance = max_edit_distance
#         self.alpha = alpha
#         self.beta = beta
#         self.gamma = gamma
#         self.vocab = set()
#         self.word_frequencies = {}
#         self.ft_model = None
#         self.fasttext_model_path = fasttext_model_path
        
#         # Spanish alphabet including diacritics
#         self.spanish_chars = set('abcdefghijklmnopqrstuvwxyzáéíóúüñ')
        
#         print("Initializing Spanish Spelling Corrector...")
#         self._load_fasttext_model()
#         if vocab_file:
#             self._load_vocabulary(vocab_file)
#         else:
#             self._create_default_vocabulary()
    
#     def _download_fasttext_model(self):
#         """Download fastText Spanish model if not available"""
#         print("Downloading fastText Spanish model (this may take a while)...")
#         try:
#             # Download compressed Spanish model
#             fasttext.util.download_model('es', if_exists='ignore')
#             return "cc.es.300.bin"
#         except Exception as e:
#             print(f"Error downloading fastText model: {e}")
#             print("Please download manually from: https://dl.fbaipublicfiles.com/fasttext/vectors-crawl/cc.es.300.bin.gz")
#             sys.exit(1)
    
#     def _load_fasttext_model(self):
#         """Load fastText Spanish embeddings model"""
#         if not os.path.exists(self.fasttext_model_path):
#             self.fasttext_model_path = self._download_fasttext_model()
        
#         try:
#             print(f"Loading fastText model from {self.fasttext_model_path}")
#             self.ft_model = fasttext.load_model(self.fasttext_model_path)
#             print("fastText model loaded successfully")
#         except Exception as e:
#             print(f"Error loading fastText model: {e}")
#             sys.exit(1)
    
#     def _create_default_vocabulary(self):
#         """Create a default Spanish vocabulary from fastText model"""
#         print("Creating default Spanish vocabulary from fastText model...")
        
#         # Get most frequent Spanish words from fastText vocabulary
#         vocab_words = self.ft_model.get_words()
        
#         # Filter for Spanish-like words (contain Spanish characters, reasonable length)
#         spanish_vocab = set()
#         for word in vocab_words[:100000]:  # Take top 100k words
#             if (2 <= len(word) <= 20 and 
#                 all(c.lower() in self.spanish_chars or c in '-' for c in word) and
#                 any(c.isalpha() for c in word)):
#                 spanish_vocab.add(word.lower())
        
#         self.vocab = spanish_vocab
#         print(f"Created vocabulary with {len(self.vocab)} words")
        
#         # Create simple frequency scores based on fastText word order
#         for i, word in enumerate(list(self.vocab)[:50000]):
#             self.word_frequencies[word] = 1.0 / (i + 1)
    
#     def _load_vocabulary(self, vocab_file: str):
#         """Load vocabulary from file"""
#         print(f"Loading vocabulary from {vocab_file}")
#         try:
#             with open(vocab_file, 'r', encoding='utf-8') as f:
#                 for line_num, line in enumerate(f, 1):
#                     line = line.strip()
#                     if line:
#                         parts = line.split('\t')
#                         word = parts[0].lower()
#                         freq = float(parts[1]) if len(parts) > 1 else 1e-6
#                         self.vocab.add(word)
#                         self.word_frequencies[word] = freq
#             print(f"Loaded {len(self.vocab)} words from vocabulary")
#         except Exception as e:
#             print(f"Error loading vocabulary: {e}")
#             self._create_default_vocabulary()
    
#     def edit_distance(self, s1: str, s2: str) -> int:
#         """Compute Levenshtein edit distance between two strings"""
#         if len(s1) < len(s2):
#             return self.edit_distance(s2, s1)
        
#         if len(s2) == 0:
#             return len(s1)
        
#         previous_row = list(range(len(s2) + 1))
#         for i, c1 in enumerate(s1):
#             current_row = [i + 1]
#             for j, c2 in enumerate(s2):
#                 insertions = previous_row[j + 1] + 1
#                 deletions = current_row[j] + 1
#                 substitutions = previous_row[j] + (c1 != c2)
#                 current_row.append(min(insertions, deletions, substitutions))
#             previous_row = current_row
        
#         return previous_row[-1]
    
#     def generate_candidates(self, word: str) -> List[str]:
#         """Generate spelling correction candidates within edit distance threshold"""
#         word = word.lower()
#         candidates = set()
        
#         # Add exact matches if in vocabulary
#         if word in self.vocab:
#             candidates.add(word)
        
#         # Generate candidates by edit operations
#         candidates.update(self._generate_edits_1(word))
#         candidates.update(self._generate_edits_2(word))
        
#         # Filter candidates that are in vocabulary
#         valid_candidates = []
#         for candidate in candidates:
#             if candidate in self.vocab:
#                 edit_dist = self.edit_distance(word, candidate)
#                 if edit_dist <= self.max_edit_distance:
#                     valid_candidates.append(candidate)
        
#         # If no valid candidates found, try approximate matching with vocabulary
#         if not valid_candidates:
#             valid_candidates = self._find_approximate_matches(word)
        
#         return valid_candidates
    
#     def _generate_edits_1(self, word: str) -> Set[str]:
#         """Generate all possible edits with distance 1"""
#         letters = self.spanish_chars
#         splits = [(word[:i], word[i:]) for i in range(len(word) + 1)]
        
#         deletes = [L + R[1:] for L, R in splits if R]
#         transposes = [L + R[1] + R[0] + R[2:] for L, R in splits if len(R) > 1]
#         replaces = [L + c + R[1:] for L, R in splits if R for c in letters]
#         inserts = [L + c + R for L, R in splits for c in letters]
        
#         return set(deletes + transposes + replaces + inserts)
    
#     def _generate_edits_2(self, word: str) -> Set[str]:
#         """Generate all possible edits with distance 2"""
#         return set(e2 for e1 in self._generate_edits_1(word) 
#                   for e2 in self._generate_edits_1(e1))
    
#     def _find_approximate_matches(self, word: str, max_candidates: int = 10) -> List[str]:
#         """Find approximate matches when no exact edit candidates exist"""
#         candidates = []
#         for vocab_word in self.vocab:
#             if abs(len(vocab_word) - len(word)) <= 2:
#                 edit_dist = self.edit_distance(word, vocab_word)
#                 if edit_dist <= self.max_edit_distance:
#                     candidates.append((vocab_word, edit_dist))
        
#         # Sort by edit distance and frequency
#         candidates.sort(key=lambda x: (x[1], -self.word_frequencies.get(x[0], 0)))
#         return [c[0] for c in candidates[:max_candidates]]
    
#     def cosine_similarity(self, vec1: np.ndarray, vec2: np.ndarray) -> float:
#         """Compute cosine similarity between two vectors"""
#         norm1 = np.linalg.norm(vec1)
#         norm2 = np.linalg.norm(vec2)
#         if norm1 == 0 or norm2 == 0:
#             return 0.0
#         return np.dot(vec1, vec2) / (norm1 * norm2)
    
#     def correct_word(self, word: str) -> str:
#         """Correct a single misspelled word"""
#         word_lower = word.lower()
        
#         # If word is already correct, return it
#         if word_lower in self.vocab:
#             return word_lower
        
#         # Generate candidates
#         candidates = self.generate_candidates(word_lower)
#         if not candidates:
#             return word_lower  # Return original if no candidates
        
#         # Get embeddings
#         input_embedding = self.ft_model.get_word_vector(word_lower)
        
#         # Score candidates
#         scored_candidates = []
#         for candidate in candidates:
#             candidate_embedding = self.ft_model.get_word_vector(candidate)
            
#             # Compute similarity scores
#             embedding_sim = self.cosine_similarity(input_embedding, candidate_embedding)
#             frequency_score = self.word_frequencies.get(candidate, 1e-8)
#             edit_dist = self.edit_distance(word_lower, candidate)
#             edit_penalty = 1.0 / (1.0 + edit_dist)
            
#             # Normalize frequency score (log scale)
#             freq_score = np.log(frequency_score + 1e-8) + 20  # Shift to positive
#             freq_score = max(0, freq_score) / 20.0  # Normalize to [0,1]
            
#             # Combined score
#             total_score = (self.alpha * embedding_sim + 
#                           self.beta * freq_score + 
#                           self.gamma * edit_penalty)
            
#             scored_candidates.append(CorrectionCandidate(
#                 word=candidate,
#                 edit_distance=edit_dist,
#                 embedding_similarity=embedding_sim,
#                 frequency_score=freq_score,
#                 total_score=total_score
#             ))
        
#         # Return best candidate
#         if scored_candidates:
#             best_candidate = max(scored_candidates, key=lambda x: x.total_score)
#             return best_candidate.word
        
#         return word_lower
    
#     def evaluate_dataset(self, dataset_path: str) -> Dict:
#         """Evaluate the spelling correction system on a dataset"""
#         print(f"Loading dataset from {dataset_path}")
        
#         try:
#             # Try reading as TSV first
#             df = pd.read_csv(dataset_path, sep='\t', encoding='utf-8')
#         except:
#             try:
#                 # Try reading as CSV
#                 df = pd.read_csv(dataset_path, encoding='utf-8')
#             except Exception as e:
#                 print(f"Error loading dataset: {e}")
#                 return {}
        
#         print(f"Dataset loaded: {len(df)} rows")
#         print(f"Columns: {list(df.columns)}")
        
#         # Ensure required columns exist
#         required_cols = ['input', 'target']
#         for col in required_cols:
#             if col not in df.columns:
#                 print(f"ERROR: Required column '{col}' not found in dataset")
#                 return {}
        
#         results = []
#         correct_predictions = 0
        
#         print("Processing corrections...")
#         for idx, row in df.iterrows():
#             input_word = str(row['input']).strip()
#             target_word = str(row['target']).strip()
            
#             # Get prediction
#             predicted = self.correct_word(input_word)
#             is_correct = predicted.lower() == target_word.lower()
            
#             if is_correct:
#                 correct_predictions += 1
            
#             result = {
#                 'input': input_word,
#                 'predicted': predicted,
#                 'target': target_word,
#                 'correct': is_correct,
#                 'source_rule': row.get('source_rule', 'unknown'),
#                 'edit_count': row.get('edit_count', -1),
#                 'freq_bucket': row.get('freq_bucket', -1)
#             }
#             results.append(result)
            
#             if (idx + 1) % 100 == 0:
#                 print(f"Processed {idx + 1}/{len(df)} rows")
        
#         # Calculate overall accuracy
#         overall_accuracy = correct_predictions / len(results)
        
#         # Calculate accuracy by source_rule
#         rule_accuracy = defaultdict(lambda: {'correct': 0, 'total': 0})
#         for result in results:
#             rule = result['source_rule']
#             rule_accuracy[rule]['total'] += 1
#             if result['correct']:
#                 rule_accuracy[rule]['correct'] += 1
        
#         # Calculate accuracy by freq_bucket
#         bucket_accuracy = defaultdict(lambda: {'correct': 0, 'total': 0})
#         for result in results:
#             bucket = result['freq_bucket']
#             bucket_accuracy[bucket]['total'] += 1
#             if result['correct']:
#                 bucket_accuracy[bucket]['correct'] += 1
        
#         evaluation_results = {
#             'overall_accuracy': overall_accuracy,
#             'total_samples': len(results),
#             'correct_predictions': correct_predictions,
#             'detailed_results': results,
#             'rule_breakdown': {rule: acc['correct']/acc['total'] 
#                              for rule, acc in rule_accuracy.items()},
#             'bucket_breakdown': {bucket: acc['correct']/acc['total'] 
#                                for bucket, acc in bucket_accuracy.items()}
#         }
        
#         return evaluation_results
    
#     def save_results(self, results: Dict, output_path: str):
#         """Save evaluation results to files"""
#         # Save detailed results as CSV
#         detailed_df = pd.DataFrame(results['detailed_results'])
#         csv_path = output_path.replace('.txt', '_detailed.csv')
#         detailed_df.to_csv(csv_path, index=False, encoding='utf-8')
        
#         # Save summary results
#         with open(output_path, 'w', encoding='utf-8') as f:
#             f.write("Spanish Spelling Correction System - Evaluation Results\n")
#             f.write("=" * 60 + "\n\n")
            
#             f.write(f"Overall Accuracy: {results['overall_accuracy']:.4f}\n")
#             f.write(f"Correct Predictions: {results['correct_predictions']}/{results['total_samples']}\n\n")
            
#             f.write("Accuracy by Source Rule:\n")
#             f.write("-" * 30 + "\n")
#             for rule, acc in results['rule_breakdown'].items():
#                 f.write(f"{rule:15}: {acc:.4f}\n")
            
#             f.write("\nAccuracy by Frequency Bucket:\n")
#             f.write("-" * 30 + "\n")
#             for bucket, acc in results['bucket_breakdown'].items():
#                 f.write(f"Bucket {bucket:2}: {acc:.4f}\n")
        
#         print(f"Results saved to {output_path}")
#         print(f"Detailed results saved to {csv_path}")

# def main():
#     parser = argparse.ArgumentParser(description="Spanish Spelling Correction System")
#     parser.add_argument('dataset', help='Path to the dataset file (TSV or CSV)')
#     parser.add_argument('--vocab', help='Path to vocabulary file (optional)')
#     parser.add_argument('--fasttext-model', default='cc.es.300.bin', 
#                        help='Path to fastText model')
#     parser.add_argument('--output', default='correction_results.txt',
#                        help='Output file for results')
#     parser.add_argument('--max-edit-distance', type=int, default=2,
#                        help='Maximum edit distance for candidates')
#     parser.add_argument('--alpha', type=float, default=0.5,
#                        help='Weight for embedding similarity')
#     parser.add_argument('--beta', type=float, default=0.3,
#                        help='Weight for word frequency')
#     parser.add_argument('--gamma', type=float, default=0.2,
#                        help='Weight for edit distance penalty')
    
#     args = parser.parse_args()
    
#     # Initialize corrector
#     corrector = SpanishSpellingCorrector(
#         vocab_file=args.vocab,
#         fasttext_model_path=args.fasttext_model,
#         max_edit_distance=args.max_edit_distance,
#         alpha=args.alpha,
#         beta=args.beta,
#         gamma=args.gamma
#     )
    
#     # Evaluate on dataset
#     results = corrector.evaluate_dataset(args.dataset)
    
#     if results:
#         # Print results
#         print("\n" + "="*60)
#         print("EVALUATION RESULTS")
#         print("="*60)
#         print(f"Overall Accuracy: {results['overall_accuracy']:.4f}")
#         print(f"Correct: {results['correct_predictions']}/{results['total_samples']}")
        
#         print("\nAccuracy by Source Rule:")
#         for rule, acc in results['rule_breakdown'].items():
#             print(f"  {rule:15}: {acc:.4f}")
        
#         print("\nAccuracy by Frequency Bucket:")
#         for bucket, acc in results['bucket_breakdown'].items():
#             print(f"  Bucket {bucket:2}: {acc:.4f}")
        
#         # Save results
#         corrector.save_results(results, args.output)
#     else:
#         print("ERROR: Could not evaluate dataset")
#         sys.exit(1)

# if __name__ == "__main__":
#     main()

usage: colab_kernel_launcher.py [-h] [--vocab VOCAB]
                                [--fasttext-model FASTTEXT_MODEL]
                                [--output OUTPUT]
                                [--max-edit-distance MAX_EDIT_DISTANCE]
                                [--alpha ALPHA] [--beta BETA] [--gamma GAMMA]
                                dataset
colab_kernel_launcher.py: error: unrecognized arguments: -f


SystemExit: 2